# Module 4 · Model Development: LoRA, QLoRA & Supervised Fine-Tuning

**Track:** Main Track — Model Development  
**Toolchain:** PyTorch · HuggingFace Transformers · PEFT · bitsandbytes · W&B  
**Prerequisites:** DL Subtrack (15_01–15_08), NLP Subtrack (16_01–16_06)  
**Difficulty:** ⭐⭐⭐ Advanced  
**Estimated Time:** 150–210 minutes

---

## Why This Notebook Is the Most Important in the Curriculum

You now understand neural networks from scratch (DL subtrack) and the full NLP pipeline from tokenization to BERT (NLP subtrack). **This notebook connects that theory to the single most valuable AI engineering skill of 2026: making a foundation model do what YOU need.**

Pre-trained LLMs are general-purpose. Fine-tuning makes them domain-specific:

| Approach | Cost | Time | Quality | When to Use |
|----------|------|------|---------|-------------|
| Prompt engineering (NB18) | $0 | Minutes | Good | Simple tasks, prototyping |
| RAG (NB22) | Low | Hours | Good for factual | When model needs external knowledge |
| **SFT with LoRA (THIS)** | $5–$100 | Hours | Excellent | Change model behavior/style/format |
| Full fine-tuning | $1K–$100K | Days | Highest | Pre-training from scratch (rare) |

**The 2026 reality:** You will almost never do full fine-tuning. LoRA/QLoRA lets you fine-tune a 70B model on a single A100 GPU by training only 0.1% of the parameters.

---

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors think fine-tuning means retraining all billions of parameters. Seniors know that LoRA decomposes the weight update into two tiny matrices ($A$ and $B$), training only 0.1–1% of the parameters while achieving 95–99% of full fine-tuning quality. This is not a hack — it's mathematically principled low-rank approximation.

**Mental Model:** Imagine a master painter (the pre-trained model) who already knows how to paint every subject. You don't retrain them from kindergarten to paint in YOUR style. Instead, you give them a thin overlay transparency sheet (LoRA adapters) and they learn to adjust their strokes on that sheet. The original painting skills are untouched; the overlay adds your customization.

**Common Junior Pitfall:** Setting `r=256` (high LoRA rank) because "more parameters = better." In practice, `r=8–32` captures the vast majority of the task-specific information. Higher ranks waste memory, increase overfitting risk, and slow training with negligible quality gains.

---

## 📑 Table of Contents

* [Why This Notebook Is the Most Important in the Curriculum](#why-this-notebook-is-the-most-important-in-the-curriculum)
  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1. The Mathematics of LoRA](#1-the-mathematics-of-lora)
  * [1.1 Low-Rank Decomposition — Why It Works](#11-low-rank-decomposition-why-it-works)
  * [1.2 The LoRA Forward Pass](#12-the-lora-forward-pass)
  * [1.3 Which Layers to Adapt?](#13-which-layers-to-adapt)
* [2. QLoRA: 4-Bit Quantization + LoRA](#2-qlora-4-bit-quantization-lora)
  * [2.1 Why QLoRA Changed Everything](#21-why-qlora-changed-everything)
  * [2.2 NF4 Quantization Explained](#22-nf4-quantization-explained)
  * [2.3 Double Quantization](#23-double-quantization)
* [3. Hands-On: Building a LoRA Module from Scratch](#3-hands-on-building-a-lora-module-from-scratch)
* [4. Production Fine-Tuning with HuggingFace PEFT](#4-production-fine-tuning-with-huggingface-peft)
  * [4.1 Loading a Quantized Base Model](#41-loading-a-quantized-base-model)
* [5. Experiment Tracking with Weights & Biases](#5-experiment-tracking-with-weights-biases)
  * [🤔 Why Track Experiments?](#why-track-experiments)
* [6. Model Merging, Export & Serialization](#6-model-merging-export-serialization)
  * [6.1 Merging LoRA Weights Back](#61-merging-lora-weights-back)
  * [6.2 SafeTensors Format](#62-safetensors-format)
  * [6.3 Pushing to HuggingFace Hub](#63-pushing-to-huggingface-hub)
* [✅ Knowledge Check](#knowledge-check)
  * [Q1: Why does LoRA initialize B to zeros?](#q1-why-does-lora-initialize-b-to-zeros)
  * [Q2: You have 500 training examples. Should you use r=8 or r=64?](#q2-you-have-500-training-examples-should-you-use-r8-or-r64)
  * [Q3: QLoRA uses 4-bit for the base model. Does training also happen in 4-bit?](#q3-qlora-uses-4-bit-for-the-base-model-does-training-also-happen-in-4-bit)
  * [Q4: Merge vs. keep separate — when to use each?](#q4-merge-vs-keep-separate-when-to-use-each)
  * [Q5: Why is the learning rate for LoRA (2e-4) much higher than full fine-tuning (2e-5)?](#q5-why-is-the-learning-rate-for-lora-2e-4-much-higher-than-full-fine-tuning-2e-5)
* [🔨 Practical Practice](#practical-practice)
  * [Tier 1: Beginner](#tier-1-beginner)
  * [Tier 2: Intermediate](#tier-2-intermediate)
  * [Tier 3: Advanced](#tier-3-advanced)
* [🎯 Summary & Key Takeaways](#summary-key-takeaways)
  * [🧠 Key Decision Framework](#key-decision-framework)


## 1. The Mathematics of LoRA

### 1.1 Low-Rank Decomposition — Why It Works

**The Key Insight (Aghajanyan et al., 2020):** When you fine-tune a large pre-trained model, the weight updates $\Delta W$ have a very low **intrinsic rank**. This means most of the information in the update can be captured by a much smaller matrix.

Standard fine-tuning updates a weight matrix $W_0 \in \mathbb{R}^{d \times k}$ by adding a full-rank update:
$$W = W_0 + \Delta W$$

where $\Delta W$ has $d \times k$ trainable parameters (millions or billions).

**LoRA's trick:** Decompose $\Delta W$ into two smaller matrices:
$$\Delta W = B \cdot A$$

where:
- $A \in \mathbb{R}^{r \times k}$ — the "down-projection" (initialized randomly)
- $B \in \mathbb{R}^{d \times r}$ — the "up-projection" (initialized to zeros)
- $r \ll \min(d, k)$ — the **rank** (typically 8–64)

**Parameter savings:**

| Model | Layer Shape ($d \times k$) | Full Fine-Tune Params | LoRA (r=16) Params | Reduction |
|-------|--------------------------|----------------------|--------------------|-----------|
| Llama 3 8B | 4096 × 4096 | 16,777,216 | 131,072 | **128×** |
| Llama 3 70B | 8192 × 8192 | 67,108,864 | 262,144 | **256×** |

### 1.2 The LoRA Forward Pass

During inference, the modified forward pass is:
$$h = W_0 x + \frac{\alpha}{r} B A x$$

Where $\alpha$ is a scaling factor (called `lora_alpha`). The ratio $\frac{\alpha}{r}$ controls the magnitude of the LoRA update.

**🤔 Why initialize B to zeros?**
At the start of training, $\Delta W = B \cdot A = 0 \cdot A = 0$. This means the model starts as the EXACT original pre-trained model. Training gradually learns the task-specific adjustment. This is critical for stability — you don't want random noise destroying the pre-trained knowledge on step 1.

### 1.3 Which Layers to Adapt?

Not all layers benefit equally from LoRA:

| Target Modules | Effect | Recommendation |
|---------------|--------|---------------|
| `q_proj`, `v_proj` | Attention queries and values | Minimum viable (original paper) |
| `q_proj`, `k_proj`, `v_proj`, `o_proj` | All attention projections | **Best default** |
| + `gate_proj`, `up_proj`, `down_proj` | Attention + MLP layers | Maximum quality, more parameters |
| `embed_tokens`, `lm_head` | Input/output embeddings | Only for vocabulary adaptation |

📈 **Production Signal:** Llama 3 fine-tuning guides recommend targeting all attention projections (`q_proj, k_proj, v_proj, o_proj`) with `r=16` as the sweet spot.

---

## 2. QLoRA: 4-Bit Quantization + LoRA

### 2.1 Why QLoRA Changed Everything

**The Problem:** Even with LoRA, you still need to load the full base model into GPU memory. Llama 3 70B in fp16 requires **140 GB** — that's 2× H100 GPUs just to load it.

**QLoRA's breakthrough (Dettmers et al., 2023):** Load the base model in **4-bit precision** (35 GB), then train LoRA adapters in fp16/bf16 on top. This gives you:

| Configuration | VRAM Needed (70B) | GPU Required | Quality vs Full FT |
|--------------|:-:|:-:|:-:|
| Full fine-tuning (fp16) | 140+ GB | 4× A100 | 100% |
| LoRA (fp16 base) | 140 GB | 2× A100 | ~99% |
| **QLoRA (4-bit base)** | **~36 GB** | **1× A100** | **~97%** |
| QLoRA (4-bit, 8B model) | **~6 GB** | **1× RTX 4090** | ~97% |

### 2.2 NF4 Quantization Explained

QLoRA uses **NormalFloat 4-bit (NF4)** — a quantization format optimized for normally-distributed neural network weights.

Standard 4-bit quantization divides the range [-max, +max] into 16 evenly spaced bins. But neural network weights follow a **bell curve** (normal distribution) — most values cluster near zero.

NF4 places more quantization bins near zero where there's more density:

```
Standard INT4:  |----|----|----|----|----|----|----|----| (uniform bins)
NF4:            |--|--|---|-----|---------|-----|---|--| (more bins near center)
                        ↑ Most weights are here
```

This reduces quantization error by ~30% compared to standard INT4.

### 2.3 Double Quantization

Each quantization group (typically 64 values) has a **scaling factor** stored in fp32 (4 bytes). For a 70B model, these scaling factors alone use ~500 MB. 

**Double quantization** quantizes the scaling factors themselves to fp8, saving an additional ~350 MB. This is enabled with `bnb_4bit_use_double_quant=True`.

---

In [ ]:
# Cell 1 — Install dependencies
# peft: Parameter-Efficient Fine-Tuning (LoRA, etc.)
# bitsandbytes: 4-bit/8-bit quantization for QLoRA
# trl: Transformer Reinforcement Learning (SFT trainer)
# wandb: Experiment tracking
!pip install -q torch transformers peft bitsandbytes trl datasets wandb accelerate

## 3. Hands-On: Building a LoRA Module from Scratch

Before using the PEFT library, let's implement LoRA ourselves to understand exactly what happens inside.

In [ ]:
# Cell 2 — LoRA from scratch in PyTorch
#
# WHAT IS HAPPENING?
# We implement the exact LoRA mechanism described in the math section:
# 1. Freeze the original weight matrix W0
# 2. Add two small trainable matrices A (down-project) and B (up-project)
# 3. Forward pass: h = W0*x + (alpha/r) * B*A*x
# 4. Only A and B are trained — W0 never changes

import torch
import torch.nn as nn
import torch.nn.functional as F

class LoRALinear(nn.Module):
    """A Linear layer with LoRA adaptation — built from scratch.
    
    This is exactly what HuggingFace PEFT does internally.
    Understanding this class = understanding LoRA.
    """
    def __init__(self, original_linear: nn.Linear, r: int = 8, lora_alpha: int = 16, lora_dropout: float = 0.05):
        super().__init__()
        self.original = original_linear
        self.original.weight.requires_grad = False  # Freeze pre-trained weights
        if self.original.bias is not None:
            self.original.bias.requires_grad = False
        
        d_out, d_in = original_linear.weight.shape
        self.r = r
        self.scaling = lora_alpha / r  # The α/r scaling factor
        
        # LoRA matrices
        self.lora_A = nn.Parameter(torch.randn(r, d_in) * 0.01)     # Down-project: d_in → r
        self.lora_B = nn.Parameter(torch.zeros(d_out, r))            # Up-project:   r → d_out
        # B initialized to ZEROS so ΔW = B·A = 0 at start
        
        self.dropout = nn.Dropout(lora_dropout)
    
    def forward(self, x):
        # Original path (frozen)
        h = self.original(x)
        
        # LoRA path (trainable)
        lora_out = self.dropout(x) @ self.lora_A.T @ self.lora_B.T * self.scaling
        
        return h + lora_out  # Combined output
    
    def merge_weights(self):
        """Merge LoRA weights back into original — for inference speed."""
        with torch.no_grad():
            self.original.weight += (self.lora_B @ self.lora_A) * self.scaling
        return self.original


# === DEMO: Apply LoRA to a pre-trained linear layer ===
torch.manual_seed(42)

# Simulate a pre-trained attention projection (4096 → 4096 like Llama)
pretrained_layer = nn.Linear(4096, 4096, bias=False)
pretrained_params = sum(p.numel() for p in pretrained_layer.parameters())

# Apply LoRA with rank 16
lora_layer = LoRALinear(pretrained_layer, r=16, lora_alpha=32)
trainable_params = sum(p.numel() for p in lora_layer.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in lora_layer.parameters())

print('🔧 LoRA Applied to Attention Projection')
print(f'  Original layer:    {pretrained_params:>12,} parameters (FROZEN)')
print(f'  LoRA A matrix:     {lora_layer.lora_A.numel():>12,} parameters (r={16} × d_in={4096})')
print(f'  LoRA B matrix:     {lora_layer.lora_B.numel():>12,} parameters (d_out={4096} × r={16})')
print(f'  Trainable params:  {trainable_params:>12,} ({trainable_params/total_params:.2%} of total)')
print(f'  Reduction factor:  {pretrained_params/trainable_params:.0f}×')

# Verify forward pass works
x = torch.randn(2, 128, 4096)  # (batch=2, seq_len=128, d_model=4096)
out = lora_layer(x)
print(f'\n  Input shape:  {list(x.shape)}')
print(f'  Output shape: {list(out.shape)}')

# Verify B=0 initialization means output starts identical to original
original_out = pretrained_layer(x)
diff = (out - original_out).abs().max().item()
print(f'\n  Max diff from original at init: {diff:.10f}')
print(f'  ✅ LoRA starts as identity (B=0 → ΔW=0 → no change)')

In [ ]:
# Cell 3 — Rank vs quality trade-off analysis
#
# 🤔 How to choose the right rank?
# Higher rank = more parameters = more expressiveness but more overfitting risk
# Lower rank = fewer parameters = faster, cheaper, but may underfit

import numpy as np

def lora_parameter_analysis(d_model, num_layers=32, num_targets=4):
    """Calculate LoRA parameter counts for different ranks.
    
    Args:
        d_model: Model hidden dimension
        num_layers: Number of transformer layers
        num_targets: Number of projection matrices per layer (q,k,v,o = 4)
    """
    base_params_per_layer = num_targets * d_model * d_model
    total_base = base_params_per_layer * num_layers
    
    print(f'Model: d_model={d_model}, layers={num_layers}, targets={num_targets}')
    print(f'Base attention params: {total_base:,} ({total_base/1e6:.0f}M)')
    print(f'\n{"Rank":>6} {"LoRA Params":>14} {"% of Base":>10} {"Est. VRAM":>12} {"Recommendation"}')
    print('─' * 75)
    
    for r in [4, 8, 16, 32, 64, 128, 256]:
        lora_per_layer = num_targets * 2 * r * d_model  # A + B matrices
        total_lora = lora_per_layer * num_layers
        pct = total_lora / total_base * 100
        vram_mb = total_lora * 2 / 1e6  # fp16 = 2 bytes
        
        if r <= 8: rec = '✅ Lightweight tasks'
        elif r <= 32: rec = '⭐ Best default'
        elif r <= 64: rec = '⚠️ Only if underfitting'
        else: rec = '❌ Rarely needed'
        
        print(f'{r:>6} {total_lora:>14,} {pct:>9.2f}% {vram_mb:>10.0f} MB  {rec}')

print('📊 LoRA Parameter Analysis for Llama 3 8B')
lora_parameter_analysis(d_model=4096, num_layers=32, num_targets=4)

print('\n' + '=' * 75)
print('\n📊 LoRA Parameter Analysis for Llama 3 70B')
lora_parameter_analysis(d_model=8192, num_layers=80, num_targets=4)

---

## 4. Production Fine-Tuning with HuggingFace PEFT

### 4.1 Loading a Quantized Base Model

In production, you use the `peft` library instead of building LoRA from scratch. Here's the complete pipeline:

```
Step 1: Load base model in 4-bit (QLoRA)     → bitsandbytes
Step 2: Attach LoRA adapters                  → peft
Step 3: Prepare training data (chat format)   → datasets
Step 4: Train with SFTTrainer                 → trl
Step 5: Log metrics                           → wandb
Step 6: Save and push adapters                → huggingface_hub
```

In [ ]:
# Cell 4 — Complete QLoRA + SFT production script
#
# This is the ACTUAL production code you'd run on a GPU.
# We print the script with detailed annotations because
# executing it requires a GPU and model download (~5GB).

production_script = '''
# ============================================================
# Production QLoRA Fine-Tuning Pipeline
# ============================================================
# Run with: python finetune_qlora.py
# Requires: GPU with >= 6GB VRAM (for 8B model)
# Time: ~2 hours for 1K examples on A100

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import wandb

# ── Step 1: Quantization Config (QLoRA) ──────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                    # Load model in 4-bit
    bnb_4bit_quant_type="nf4",            # NormalFloat4 (best for LLMs)
    bnb_4bit_compute_dtype=torch.bfloat16, # Compute in bf16 (faster)
    bnb_4bit_use_double_quant=True,        # Save ~350MB extra
)

# ── Step 2: Load Base Model + Tokenizer ─────────────────────
model_name = "meta-llama/Llama-3.1-8B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",           # Automatically distribute across GPUs
    attn_implementation="flash_attention_2",  # 2-4x faster (NB30)
)
model = prepare_model_for_kbit_training(model)  # Enable gradient checkpointing

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # Required for batched training
tokenizer.padding_side = "right"

# ── Step 3: LoRA Configuration ──────────────────────────────
lora_config = LoraConfig(
    r=16,                          # Rank — 16 is the sweet spot
    lora_alpha=32,                 # Scaling factor (alpha/r = 2.0)
    lora_dropout=0.05,             # Regularization
    target_modules=[               # Which layers get LoRA
        "q_proj", "k_proj", "v_proj", "o_proj",  # All attention
    ],
    bias="none",                   # Don't train biases
    task_type="CAUSAL_LM",         # Autoregressive language model
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Output: trainable params: 6,553,600 || all params: 8,036,528,128 || trainable%: 0.0816

# ── Step 4: Prepare Training Data ───────────────────────────
# Format: ChatML / Llama chat template
dataset = load_dataset("your-org/sft-data", split="train")

def format_chat(example):
    """Convert to Llama chat format."""
    messages = [
        {"role": "system", "content": example["system_prompt"]},
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["response"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

dataset = dataset.map(format_chat)

# ── Step 5: Training Configuration ──────────────────────────
wandb.init(project="llm-finetuning", name="llama3-8b-qlora-sft")

training_args = SFTConfig(
    output_dir="./checkpoints/llama3-sft",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,   # Effective batch = 4*4 = 16
    learning_rate=2e-4,              # Higher LR for LoRA (vs 2e-5 for full FT)
    lr_scheduler_type="cosine",      # Cosine annealing
    warmup_ratio=0.03,              # 3% warmup
    max_seq_length=2048,
    bf16=True,                       # bfloat16 training
    logging_steps=10,
    save_steps=100,
    save_total_limit=3,             # Keep only 3 checkpoints
    report_to="wandb",
    gradient_checkpointing=True,     # Trade compute for memory
    dataset_text_field="text",
)

# ── Step 6: Train! ──────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

trainer.train()

# ── Step 7: Save LoRA adapters (NOT the full model) ─────────
trainer.save_model("./final-adapters")  # Only ~25MB (just A and B matrices)
# Compare: full model = 16GB. LoRA adapters = 25MB. That\'s 640× smaller.
'''

print('🚀 Production QLoRA + SFT Fine-Tuning Pipeline')
print(production_script)

print('\n📊 Key Hyperparameters Explained:')
print('  r=16         → LoRA rank. Start here. Go to 32 only if quality is poor.')
print('  alpha=32     → Scaling factor. Rule of thumb: alpha = 2 * r')
print('  lr=2e-4      → Much higher than full fine-tuning (which uses 2e-5)')
print('  epochs=3     → SFT typically needs 1-3 epochs. More → overfitting.')
print('  batch=4×4=16 → Gradient accumulation simulates larger batch on small GPU')

In [ ]:
# Cell 5 — Training data format examples
#
# The #1 cause of bad fine-tuning is BAD DATA, not bad hyperparameters.
# Here are the formats that work.

import json

# === Format 1: Instruction-Response (Alpaca style) ===
alpaca_example = {
    "instruction": "Write a SQL query to find the top 5 customers by revenue.",
    "input": "Tables: customers(id, name, email), orders(id, customer_id, amount, date)",
    "output": """SELECT c.name, SUM(o.amount) as total_revenue
FROM customers c
JOIN orders o ON c.id = o.customer_id
GROUP BY c.id, c.name
ORDER BY total_revenue DESC
LIMIT 5;"""
}

# === Format 2: ChatML (for chat models — RECOMMENDED) ===
chatml_example = {
    "messages": [
        {"role": "system", "content": "You are a senior data engineer. Write clean, optimized SQL."},
        {"role": "user", "content": "Find top 5 customers by revenue. Tables: customers(id, name), orders(id, customer_id, amount)"},
        {"role": "assistant", "content": "```sql\nSELECT c.name, SUM(o.amount) AS total_revenue\nFROM customers c\nJOIN orders o ON c.id = o.customer_id\nGROUP BY c.id, c.name\nORDER BY total_revenue DESC\nLIMIT 5;\n```\n\nThis query joins customers with their orders, aggregates revenue, and returns the top 5."}
    ]
}

print('📦 Training Data Formats')
print('=' * 60)
print('\n1️⃣ Alpaca Format (instruction-tuning):')
print(json.dumps(alpaca_example, indent=2)[:300])
print(f'\n2️⃣ ChatML Format (chat fine-tuning — RECOMMENDED):')
print(json.dumps(chatml_example, indent=2)[:400])

print(f'\n💡 Data Quality Rules:')
print(f'  1. Quality > Quantity: 1K great examples > 100K noisy ones')
print(f'  2. Diversity: cover edge cases, not just the happy path')
print(f'  3. Consistency: same format, same style across all examples')
print(f'  4. Decontamination: remove any test set overlap')
print(f'  5. Length: match the output length you want at inference')

---

## 5. Experiment Tracking with Weights & Biases

### 🤔 Why Track Experiments?

Without tracking, fine-tuning becomes "I tried some stuff and this one seemed good."

| What to Track | Why |
|--------------|-----|
| Hyperparameters (r, alpha, lr) | Reproduce best results |
| Training loss curve | Detect overfitting early |
| Evaluation samples | Qualitative quality check |
| GPU utilization / VRAM | Optimize cost |
| Final model card | Document what you shipped |

In [ ]:
# Cell 6 — W&B experiment tracking simulation

import json
import numpy as np

class ExperimentTracker:
    """Simulates W&B experiment tracking for fine-tuning runs."""
    
    def __init__(self, project, name, config):
        self.project = project
        self.name = name
        self.config = config
        self.metrics = []
    
    def log(self, step, metrics):
        metrics['step'] = step
        self.metrics.append(metrics)
    
    def summary(self):
        final = self.metrics[-1] if self.metrics else {}
        return {
            'run_name': self.name,
            'final_loss': final.get('loss', 'N/A'),
            'best_loss': min(m.get('loss', 999) for m in self.metrics),
            'total_steps': len(self.metrics),
            'config': self.config,
        }

# Simulate 3 fine-tuning runs with different LoRA ranks
np.random.seed(42)
experiments = [
    {'name': 'llama3-lora-r8',  'config': {'r': 8,  'alpha': 16, 'lr': 2e-4}},
    {'name': 'llama3-lora-r16', 'config': {'r': 16, 'alpha': 32, 'lr': 2e-4}},
    {'name': 'llama3-lora-r64', 'config': {'r': 64, 'alpha': 128,'lr': 1e-4}},
]

print('📊 W&B Experiment Comparison')
print('=' * 70)

for exp in experiments:
    tracker = ExperimentTracker('llm-finetuning', exp['name'], exp['config'])
    
    # Simulate training loss curve (lower rank converges faster but plateaus higher)
    base_loss = 2.5
    r = exp['config']['r']
    min_loss = 0.8 - (r / 200)  # Higher rank → slightly lower final loss
    
    for step in range(100):
        decay = np.exp(-step / (20 + r/4))  # Higher rank = slower convergence
        noise = np.random.normal(0, 0.02)
        loss = min_loss + (base_loss - min_loss) * decay + noise
        tracker.log(step, {'loss': max(0.1, loss), 'lr': exp['config']['lr'] * (1 - step/100)})
    
    s = tracker.summary()
    print(f"\n  🔬 {s['run_name']}")
    print(f"     Config: r={exp['config']['r']}, alpha={exp['config']['alpha']}, lr={exp['config']['lr']}")
    print(f"     Final loss: {s['final_loss']:.4f}")
    print(f"     Best loss:  {s['best_loss']:.4f}")
    lora_params = 32 * 4 * 2 * r * 4096  # layers * targets * (A+B) * r * d_model
    print(f"     LoRA params: {lora_params:,} ({lora_params/8e9:.3%} of 8B)")

print(f'\n💡 Key observation: r=16 achieves nearly the same loss as r=64')
print(f'   but uses 4× fewer parameters. This is the sweet spot.')
print(f'\n   In W&B: wandb.init(project="llm-finetuning", config={{"r": 16, ...}})')
print(f'   All metrics auto-logged by SFTTrainer with report_to="wandb"')

---

## 6. Model Merging, Export & Serialization

### 6.1 Merging LoRA Weights Back

During training you keep LoRA adapters separate (~25MB). For deployment, you have two options:

| Option | Adapter Size | Inference Speed | Flexibility |
|--------|:-:|:-:|:-:|
| Keep separate | 25 MB | Slightly slower (extra matmul) | Can swap adapters per request |
| **Merge into base** | Full model size | Fastest (single matmul) | Fixed — one model |

**When to merge:** Production deployment where you serve one model. **When to keep separate:** Multi-tenant systems where different customers need different behaviors (load different adapters per request).

### 6.2 SafeTensors Format

**Never use pickle (.bin) for model serialization in production.** Pickle files can execute arbitrary code when loaded (security vulnerability). **SafeTensors** is the industry standard:

| Format | Security | Load Speed | Used By |
|--------|:--------:|:----------:|--------|
| `.bin` (pickle) | ❌ Unsafe | Slow | Legacy |
| `.safetensors` | ✅ Safe | 2-5× faster | HuggingFace default |
| `.gguf` | ✅ Safe | Fast | llama.cpp (NB31) |

### 6.3 Pushing to HuggingFace Hub

```python
# Save merged model in SafeTensors format
merged_model = model.merge_and_unload()  # Merge LoRA → base
merged_model.save_pretrained("./merged-model", safe_serialization=True)
tokenizer.save_pretrained("./merged-model")

# Push to HuggingFace Hub (private by default)
merged_model.push_to_hub("your-org/llama3-8b-custom-sft", private=True)
tokenizer.push_to_hub("your-org/llama3-8b-custom-sft", private=True)
```

In [ ]:
# Cell 7 — Demonstrate weight merging with our scratch LoRA

print('🔀 LoRA Weight Merging Demo')
print('=' * 60)

# Simulate training (update LoRA matrices)
with torch.no_grad():
    lora_layer.lora_A.normal_(0, 0.1)  # Simulate trained A
    lora_layer.lora_B.normal_(0, 0.01) # Simulate trained B

# Before merge: two forward passes (original + LoRA)
x_test = torch.randn(1, 10, 4096)
out_before = lora_layer(x_test)

# Merge weights
merged = lora_layer.merge_weights()
out_after = merged(x_test)

# Verify they're identical
max_diff = (out_before - out_after).abs().max().item()
print(f'  Output before merge: {out_before.shape}')
print(f'  Output after merge:  {out_after.shape}')
print(f'  Max difference: {max_diff:.8f}')
print(f'  ✅ Outputs are identical (merge is lossless)')

print(f'\n  Before merge: 2 matrix multiplications per forward pass')
print(f'  After merge:  1 matrix multiplication (faster inference)')
print(f'\n  Adapter size (LoRA only):  ~{(lora_layer.lora_A.numel() + lora_layer.lora_B.numel()) * 2 / 1e6:.1f} MB')
print(f'  Merged model size:         ~{merged.weight.numel() * 2 / 1e6:.0f} MB')

print(f'\n📦 Serialization Best Practices:')
print(f'  1. Always use SafeTensors: model.save_pretrained(path, safe_serialization=True)')
print(f'  2. Save adapters separately for multi-tenant serving')
print(f'  3. Push to HuggingFace Hub as private repo for team access')
print(f'  4. Include a model card with training details and eval metrics')

---
## ✅ Knowledge Check

### Q1: Why does LoRA initialize B to zeros?
<details><summary>👉 Answer</summary>

If B=0, then ΔW = B·A = 0, meaning the model starts IDENTICAL to the pre-trained model. This ensures training stability — the model doesn't lose its pre-trained knowledge on step 1. The LoRA update is learned gradually during training. If both A and B were randomly initialized, the random ΔW would corrupt the pre-trained weights immediately.
</details>

### Q2: You have 500 training examples. Should you use r=8 or r=64?
<details><summary>👉 Answer</summary>

Use **r=8** (or even r=4). With only 500 examples, a high-rank LoRA (r=64) has too many trainable parameters relative to the dataset size, leading to severe **overfitting**. The model would memorize the 500 examples instead of learning generalizable patterns. Lower rank acts as implicit regularization.
</details>

### Q3: QLoRA uses 4-bit for the base model. Does training also happen in 4-bit?
<details><summary>👉 Answer</summary>

**No.** The base model weights are STORED in 4-bit but are **dequantized to bf16 for computation**. The LoRA adapter weights (A and B) are always in fp16/bf16. Gradients are computed in bf16. Only the frozen base weights are quantized — the trainable LoRA weights maintain full precision. This is why QLoRA achieves near-full-precision quality.
</details>

### Q4: Merge vs. keep separate — when to use each?
<details><summary>👉 Answer</summary>

**Merge** when deploying a single model for all users — it's faster (one matmul instead of two) and simpler. **Keep separate** in multi-tenant systems where different customers need different fine-tuned behaviors — load the base model once, then swap LoRA adapters per request with near-zero latency (only ~25MB to load).
</details>

### Q5: Why is the learning rate for LoRA (2e-4) much higher than full fine-tuning (2e-5)?
<details><summary>👉 Answer</summary>

LoRA trains far fewer parameters (0.1% of the model), so each parameter update must be larger to have a meaningful effect on the model's output. The `alpha/r` scaling factor partially compensates, but a higher base learning rate is still needed. Full fine-tuning uses a lower LR because updating ALL parameters simultaneously risks catastrophic forgetting of pre-trained knowledge.
</details>

---
## 🔨 Practical Practice

### Tier 1: Beginner
1. Modify the `LoRALinear` class to accept a `target_modules` list and automatically apply LoRA to matching layers in any `nn.Module`.
2. Calculate the total trainable parameters for QLoRA fine-tuning of Mistral 7B (d_model=4096, 32 layers) with r=32, targeting q_proj, k_proj, v_proj, o_proj.

### Tier 2: Intermediate
1. **Rank Ablation Study:** Using the production script as a template, design an experiment that trains 3 runs with r=4, r=16, r=64 on the same dataset. Track loss curves in W&B and identify the optimal rank.
2. **Chat Template Engineering:** Write a function that converts Alpaca format (`{instruction, input, output}`) to Llama 3 ChatML format with proper `<|begin_of_text|>` tokens. Verify token counts with `tiktoken`.

### Tier 3: Advanced
1. **LoRA Arithmetic:** Research and implement LoRA weight merging of TWO different adapters trained on different tasks. Show that `adapter_sql + adapter_python` produces a model that can do both SQL and Python generation.
2. **Memory Optimization:** Implement gradient checkpointing in the from-scratch LoRA training loop. Measure peak VRAM before and after, and explain the compute-memory trade-off.

---

## 🎯 Summary & Key Takeaways

| Concept | What You Learned | Production Impact |
|---------|-----------------|------------------|
| LoRA theory | Low-rank decomposition of weight updates | Train 0.1% of params, get 99% of quality |
| QLoRA | 4-bit base + fp16 adapters | Fine-tune 70B on 1 GPU |
| PEFT library | Production LoRA implementation | 5 lines to add LoRA to any model |
| SFT training | Supervised fine-tuning pipeline | The core skill of AI engineering |
| Experiment tracking | W&B integration | Reproducibility and comparison |
| Model serialization | SafeTensors, merge, Hub push | Secure, fast model deployment |

### 🧠 Key Decision Framework

```
Need to change model KNOWLEDGE?  → Use RAG (NB22), not fine-tuning
Need to change model BEHAVIOR?   → Use SFT with LoRA (THIS notebook)
Need to change model PREFERENCES? → Use DPO alignment (NB17 — NEXT)
```

**Next →** `24_01_alignment_dpo_data_prep.ipynb` — Alignment, RLHF/DPO & Fine-Tuning Data Preparation